# 🚀 Train Model — Dự đoán Lượt Bán Sản Phẩm

**Bài toán:** Regression — dự đoán `Sales Count` từ thông tin giá, rating, discount và sentiment review.

**Pipeline:**
```
Load → Clean → Feature Engineering → Split → Scale → Hyperparameter Tuning → Evaluate → Compare
```

> ⚠️ Chỉ dùng features là **nguyên nhân** của doanh số — không dùng `Number Reviews` (target leakage).


In [ ]:
# ── Cell 1: Imports ────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, time

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import randint, uniform, loguniform

try:
    import xgboost as xgb
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print('⚠️  XGBoost chưa cài. Chạy: pip install xgboost')

try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    HAS_LGB = False
    print('⚠️  LightGBM chưa cài. Chạy: pip install lightgbm')

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)
np.random.seed(42)
print('✅ Imports OK')

In [ ]:
# ── Cell 2: Load & Inspect ─────────────────────────────────────────────
FILE_PATH    = r'D:\final_dataset.csv'
TARGET       = 'Sales Count'
RANDOM_STATE = 42
TEST_SIZE    = 0.2
CV_FOLDS     = 5
N_ITER       = 50    # số lần thử random search (tăng để tìm tốt hơn, giảm để nhanh)

df = pd.read_csv(FILE_PATH)
print(f'Shape: {df.shape}')
df.head(3)

In [ ]:
# ── Cell 3: Clean ─────────────────────────────────────────────────────
df.dropna(inplace=True)
df.drop_duplicates(inplace=True)

Q1, Q3 = df[TARGET].quantile(0.25), df[TARGET].quantile(0.75)
IQR = Q3 - Q1
before = len(df)
df = df[(df[TARGET] >= Q1 - 3*IQR) & (df[TARGET] <= Q3 + 3*IQR)]
print(f'✅ Loại {before - len(df)} outlier | Shape: {df.shape}')

In [ ]:
# ── Cell 4: Feature Engineering ───────────────────────────────────────
df.drop(columns=[c for c in ['product_id'] if c in df.columns], inplace=True)

total_sentiment = df['pos_count'] + df['neg_count'] + df['neu_count'] + 1
df['pos_ratio'] = df['pos_count'] / total_sentiment
df['neg_ratio'] = df['neg_count'] / total_sentiment

df['log_price']        = np.log1p(df['Price (After Discount)'])
df['price_x_discount'] = df['log_price']      * df['Discount Rate']
df['rating_x_pos']     = df['Rating Average'] * df['pos_ratio']
df['discount_x_pos']   = df['Discount Rate']  * df['pos_ratio']
df['log_sales']        = np.log1p(df[TARGET])

FEATURE_COLS = [
    'log_price', 'Discount Rate', 'Rating Average',
    'pos_ratio', 'neg_ratio',
    'price_x_discount', 'rating_x_pos', 'discount_x_pos',
]

X     = df[FEATURE_COLS].copy()
y     = df['log_sales'].copy()
y_raw = df[TARGET].copy()
print(f'✅ {len(FEATURE_COLS)} features | {len(X)} samples')

In [ ]:
# ── Cell 5: Split → Scale ─────────────────────────────────────────────
X_train, X_test, y_train, y_test, y_train_raw, y_test_raw = train_test_split(
    X, y, y_raw, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

scaler = MinMaxScaler()
X_train_s = pd.DataFrame(scaler.fit_transform(X_train), columns=FEATURE_COLS, index=X_train.index)
X_test_s  = pd.DataFrame(scaler.transform(X_test),      columns=FEATURE_COLS, index=X_test.index)

print(f'Train: {X_train_s.shape} | Test: {X_test_s.shape}')
print('✅ Sẵn sàng tuning!')

---
## 🔧 Hyperparameter Tuning với RandomizedSearchCV

**Tại sao RandomizedSearch thay vì GridSearch?**
- Grid Search: thử **mọi tổ hợp** → rất chậm khi nhiều tham số
- Randomized Search: thử **N_ITER tổ hợp ngẫu nhiên** → nhanh hơn ~10x, kết quả xấp xỉ tương đương

**Scoring:** `neg_mean_squared_error` trên log scale → tương đương tối ưu R²

In [ ]:
# ── Helper: Đánh giá model ────────────────────────────────────────────
results = {}

def tune_and_evaluate(name, base_model, param_dist, X_tr, X_te, y_tr, y_te, y_te_raw):
    print(f'\n🔍 Tuning {name} ({N_ITER} iterations × {CV_FOLDS}-fold CV)...')
    t0 = time.time()

    search = RandomizedSearchCV(
        base_model,
        param_distributions=param_dist,
        n_iter=N_ITER,
        cv=CV_FOLDS,
        scoring='neg_mean_squared_error',
        n_jobs=-1,
        random_state=RANDOM_STATE,
        verbose=0
    )
    search.fit(X_tr, y_tr)
    elapsed = time.time() - t0

    best = search.best_estimator_
    y_pred_log = best.predict(X_te)
    y_pred_raw = np.clip(np.expm1(y_pred_log), 0, None)

    cv_r2  = cross_val_score(best, X_tr, y_tr, cv=CV_FOLDS, scoring='r2', n_jobs=-1).mean()
    r2     = r2_score(y_te, y_pred_log)
    mae    = mean_absolute_error(y_te_raw, y_pred_raw)
    rmse   = np.sqrt(mean_squared_error(y_te_raw, y_pred_raw))

    results[name] = {
        'CV R²': cv_r2, 'Test R²': r2, 'MAE': mae, 'RMSE': rmse,
        'model': best, 'best_params': search.best_params_,
        'y_pred_raw': y_pred_raw
    }

    print(f'  ⏱  {elapsed:.1f}s')
    print(f'  Best params: {search.best_params_}')
    print(f'  CV R²   (train) : {cv_r2:.4f}')
    print(f'  Test R²         : {r2:.4f}')
    print(f'  MAE  (lượt bán) : {mae:,.1f}')
    print(f'  RMSE (lượt bán) : {rmse:,.1f}')
    return best

In [ ]:
# ── Cell 6: Tune Ridge Regression ─────────────────────────────────────
ridge_params = {
    'alpha': loguniform(1e-3, 1e3),    # tìm alpha tối ưu trên log-scale [0.001 → 1000]
}

tune_and_evaluate(
    'Ridge Regression',
    Ridge(),
    ridge_params,
    X_train_s, X_test_s, y_train, y_test, y_test_raw
)

In [ ]:
# ── Cell 7: Tune Random Forest ─────────────────────────────────────────
rf_params = {
    'n_estimators'  : randint(200, 800),         # số cây
    'max_depth'     : [None, 6, 8, 10, 15, 20],  # None = không giới hạn
    'min_samples_leaf': randint(2, 20),           # min samples tại leaf
    'max_features'  : ['sqrt', 'log2', 0.5, 0.7],# số features xét mỗi split
    'bootstrap'     : [True, False],
}

tune_and_evaluate(
    'Random Forest',
    RandomForestRegressor(n_jobs=-1, random_state=RANDOM_STATE),
    rf_params,
    X_train_s, X_test_s, y_train, y_test, y_test_raw
)

In [ ]:
# ── Cell 8: Tune XGBoost ───────────────────────────────────────────────
if HAS_XGB:
    xgb_params = {
        'n_estimators'    : randint(200, 1000),
        'learning_rate'   : loguniform(0.01, 0.3),  # [0.01 → 0.3]
        'max_depth'       : randint(3, 10),
        'subsample'       : uniform(0.6, 0.4),       # [0.6 → 1.0]
        'colsample_bytree': uniform(0.6, 0.4),
        'reg_alpha'       : loguniform(1e-4, 10),    # L1
        'reg_lambda'      : loguniform(1e-4, 10),    # L2
        'min_child_weight': randint(1, 10),
        'gamma'           : uniform(0, 0.5),
    }
    tune_and_evaluate(
        'XGBoost',
        xgb.XGBRegressor(random_state=RANDOM_STATE, verbosity=0, n_jobs=-1),
        xgb_params,
        X_train_s, X_test_s, y_train, y_test, y_test_raw
    )
else:
    print('⏭  Bỏ qua XGBoost (chưa cài). Chạy: pip install xgboost')

In [ ]:
# ── Cell 9: Tune LightGBM ──────────────────────────────────────────────
if HAS_LGB:
    lgb_params = {
        'n_estimators'    : randint(200, 1000),
        'learning_rate'   : loguniform(0.01, 0.3),
        'num_leaves'      : randint(16, 128),       # độ phức tạp cây
        'max_depth'       : randint(3, 12),
        'min_child_samples': randint(10, 50),       # tránh overfit
        'subsample'       : uniform(0.6, 0.4),
        'colsample_bytree': uniform(0.6, 0.4),
        'reg_alpha'       : loguniform(1e-4, 10),
        'reg_lambda'      : loguniform(1e-4, 10),
    }
    tune_and_evaluate(
        'LightGBM',
        lgb.LGBMRegressor(random_state=RANDOM_STATE, verbose=-1, n_jobs=-1),
        lgb_params,
        X_train_s, X_test_s, y_train, y_test, y_test_raw
    )
else:
    print('⏭  Bỏ qua LightGBM (chưa cài). Chạy: pip install lightgbm')

In [ ]:
# ── Cell 10: So sánh kết quả ───────────────────────────────────────────
compare_cols = ['CV R²', 'Test R²', 'MAE', 'RMSE']
df_results = pd.DataFrame(
    {k: {m: results[k][m] for m in compare_cols} for k in results}
).T.sort_values('Test R²', ascending=False)

print('\n📊 Bảng so sánh sau tuning:')
print(df_results.to_string())

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
palette = ['#2ecc71', '#3498db', '#e67e22', '#9b59b6']

for ax, metric in zip(axes, ['Test R²', 'MAE', 'RMSE']):
    ascending = metric in ['MAE', 'RMSE']
    data = df_results[metric].sort_values(ascending=ascending)
    bars = ax.barh(data.index, data.values,
                   color=palette[:len(data)], edgecolor='white', linewidth=0.5)
    ax.set_title(metric, fontsize=13, fontweight='bold')
    ax.set_xlabel(metric)
    for bar, val in zip(bars, data.values):
        ax.text(bar.get_width() * 0.02, bar.get_y() + bar.get_height()/2,
                f'{val:,.3f}', va='center', fontsize=10)

plt.suptitle('Model Comparison (After Hyperparameter Tuning)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

best_name = df_results.index[0]
print(f'\n🏆 Model tốt nhất: {best_name}')
print(f'   Best params: {results[best_name]["best_params"]}')

In [ ]:
# ── Cell 11: Feature Importance (best model) ───────────────────────────
best_model = results[best_name]['model']

if hasattr(best_model, 'feature_importances_'):
    imp = pd.Series(best_model.feature_importances_, index=FEATURE_COLS).sort_values()
    colors_bar = ['#e74c3c' if v == imp.max() else '#3498db' for v in imp.values]

    fig, ax = plt.subplots(figsize=(8, 5))
    imp.plot(kind='barh', color=colors_bar, ax=ax)
    ax.set_title(f'Feature Importance — {best_name}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Importance Score')
    plt.tight_layout()
    plt.show()

    print('\n📌 Top features:')
    print(imp.sort_values(ascending=False).to_string())

elif hasattr(best_model, 'coef_'):
    coef = pd.Series(np.abs(best_model.coef_), index=FEATURE_COLS).sort_values()
    coef.plot(kind='barh', color='#3498db', figsize=(8, 5))
    plt.title(f'|Coefficients| — {best_name}', fontsize=13)
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Cell 12: Actual vs Predicted + Residuals (best model) ─────────────
y_pred_best = results[best_name]['y_pred_raw']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter
ax = axes[0]
ax.scatter(y_test_raw, y_pred_best, alpha=0.4, s=20, color='#3498db')
lim = max(y_test_raw.max(), y_pred_best.max())
ax.plot([0, lim], [0, lim], 'r--', lw=1.5, label='Perfect fit')
ax.set_xlabel('Actual Sales Count')
ax.set_ylabel('Predicted Sales Count')
ax.set_title(f'Actual vs Predicted — {best_name}')
ax.legend()

# Residuals
ax = axes[1]
residuals = y_test_raw.values - y_pred_best
ax.hist(residuals, bins=50, color='#e74c3c', alpha=0.75, edgecolor='white')
ax.axvline(0, color='black', linestyle='--', lw=1.5)
ax.set_xlabel('Residual (Actual − Predicted)')
ax.set_ylabel('Count')
ax.set_title('Residual Distribution')

plt.suptitle(f'{best_name} — Final Evaluation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

r = results[best_name]
print(f'\n📊 Final Results — {best_name}:')
print(f'  CV R²  (train/5-fold) = {r["CV R²"]:.4f}')
print(f'  Test R²               = {r["Test R²"]:.4f}')
print(f'  MAE   (lượt bán)      = {r["MAE"]:,.1f}')
print(f'  RMSE  (lượt bán)      = {r["RMSE"]:,.1f}')